# 🌿 Sprint 1 — Task 2: Triple LST + NO₂ + NDVI Visualization for CDMX

**Project:** *Dos Méxicos Bajo el Mismo Sol* — Urban heat islands, air quality, and environmental justice in CDMX.

**Notebook goal:** Add **NDVI** (vegetation index) as a third layer to the LST + NO₂ map from Task 1. The three layers together let us visually inspect the *triple environmental inequality*: sparse vegetation, hotter surfaces, and worse air quality in the same areas.

**Architecture:** Same as notebook 02 — all logic lives in the `src/` package. This notebook is a thin *pipeline orchestrator*.

---
### 🧠 Signal-processing cheat-sheet for Task 2

| GEE concept | Signal-processing analog |
|---|---|
| `normalizedDifference([NIR, RED])` | Normalized cross-correlation between two spectral channels |
| NDVI = (NIR − RED) / (NIR + RED) | Coherence — high when channels co-vary, low when they don't |
| NDVI ≈ 0.7 (forest) vs 0.1 (urban) | High vegetation coherence vs low vegetation coherence |
| Triple-layer stacking | Multi-channel display (like an RGB false-color composite) |

### 🧠 The environmental-justice hypothesis in one image

If the hypothesis holds, the three layers should *spatially correlate*:
- **Low NDVI** (red/bare soil) → **high LST** (red/hot) → **high NO₂** (red/polluted)
- **High NDVI** (green/forests) → **low LST** (blue/cool) → **low NO₂** (green/clean)

Toggle layers in the control panel to verify visually before we move to Sprint 2 (quantitative correlation).

In [1]:
# ---- Imports & Earth Engine initialization ----
# The reload block below forces Python to re-read the latest src/*.py
# from disk on every run, so editing a helper function and re-running
# this cell picks up the change without needing a kernel restart.
import importlib
import ee

import src
import src.aoi
import src.config
import src.landsat
import src.sentinel5p
import src.visualization
for mod in (src, src.aoi, src.config, src.landsat, src.sentinel5p, src.visualization):
    importlib.reload(mod)

from src import (
    EE_PROJECT_ID,
    build_triple_map,
    get_cdmx_aoi,
    load_lst_composite,
    load_ndvi_composite,
    load_no2_composite,
)

# Replace with your GCP project ID that has the Earth Engine API enabled.
ee.Initialize(project=EE_PROJECT_ID)
print("✅ Earth Engine initialized")

/home/nells-it/Documentos/PERSONAL/Portafolio/Project1-IslasCalor/project1.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✅ Earth Engine initialized


In [2]:
# ---- Load the CDMX AOI from the INEGI Marco Geoestadistico shapefile ----
aoi_cdmx = get_cdmx_aoi()
area_km2 = aoi_cdmx.geometry().area().divide(1e6).getInfo()
print(f"CDMX AOI area: {area_km2:,.1f} km²")

CDMX AOI area: 1,489.6 km²


In [3]:
# ---- Build the three composites ----
# LST: Landsat 8 Collection 2 L2, summer 2023 (June-August), cloud < 20%.
lst_summer_2023 = load_lst_composite(
    aoi_cdmx,
    start_date="2023-06-01",
    end_date="2023-09-01",
    cloud_max=20.0,
)
print("LST composite ready:", lst_summer_2023.bandNames().getInfo())

# NO₂: Sentinel-5P TROPOMI, full year 2023, cloud_fraction < 0.3.
no2_2023 = load_no2_composite(
    aoi_cdmx,
    start_date="2023-01-01",
    end_date="2023-12-31",
    cloud_max=0.3,
)
print("NO₂ composite ready:", no2_2023.bandNames().getInfo())

# NDVI: Landsat 8 Collection 2 L2, summer 2023, same cloud filter as LST.
ndvi_summer_2023 = load_ndvi_composite(
    aoi_cdmx,
    start_date="2023-06-01",
    end_date="2023-09-01",
    cloud_max=20.0,
)
print("NDVI composite ready:", ndvi_summer_2023.bandNames().getInfo())

LST composite ready: ['LST_C']
NO₂ composite ready: ['NO2_mol_m2']
NDVI composite ready: ['NDVI']


In [4]:
# ---- Build the interactive triple-layer map ----
# Layers (in stacking order, bottom -> top of the map):
#   1. NDVI — Summer 2023
#   2. NO₂  — 2023 annual median
#   3. LST  — Summer 2023 median
#   4. CDMX AOI outline (white, on top so it's always visible)
Map = build_triple_map(lst_summer_2023, no2_2023, ndvi_summer_2023, aoi_cdmx)
Map

Map(center=[19.4326, -99.1332], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topr…

### ✅ Checkpoint — what just happened

- **AOI:** CDMX state boundary, area ~1,485 km².
- **LST composite:** cloud-masked median, Jun–Aug 2023, in °C.
- **NO₂ composite:** annual median 2023, in mol/m².
- **NDVI composite:** cloud-masked median, Jun–Aug 2023, unitless [-1, 1].
- **Map:** three toggleable raster layers + AOI outline, each with its own colorbar.

### 👀 What to look for in the map

Toggle layers and compare:
1. **NDVI only** — do you see the green belt on the south (Ajusco, Desierto de los Leones)? The red core (Centro, Iztapalapa)?
2. **LST only** — does the hot pattern mirror the low-NDVI pattern?
3. **NO₂ only** — does the polluted pattern follow the urban core?

If the patterns are visually correlated, our environmental-justice hypothesis is on track for Sprint 2's quantitative test.

### ⚡ MVP check

If the triple map renders, you're done with Sprint 1, Task 2. Don't worry about:
- Fine-tuning the colorbar ranges — defaults are tuned for CDMX summer; tweak `NDVI_VIS_PARAMS` in `src/config.py` if the contrast feels off.
- Computing correlations quantitatively — that's Sprint 2 (pandas + scipy + PySAL).
- Caching the composites to disk — we'll add that in Sprint 1, Task 3.

### 🔜 Next step (Sprint 1, Task 3)

**Zonal statistics over AGEBs.** Convert the three rasters into a per-AGEB table (one row per census block, one column per metric). This unlocks Sprint 2's correlation analysis. Two small additions:

1. New function `load_ageb_aoi()` in `src/aoi.py` — points at `09a.shp` (the AGEB shapefile) instead of `09ent.shp` (state boundary).
2. New module `src/zonal.py` with `zonal_stats(image, features, statistic='mean')` using `ee.Image.reduceRegions`.
3. One new notebook `04_zonal_stats_export.ipynb` that joins the three rasters + the AGEB attributes into a single `pandas.DataFrame` and saves it to `outputs/cdmx_master_2023.parquet`.